In [2]:
#pip install torch==2.0.0+cu117 -f https://download.pytorch.org/whl/torch_stable.html
import anndata
import numpy as np
import scvelo as scv
import scanpy as sc
import torch
import os.path
import time
import latentvelo as ltv
import pickle as pickle
import matplotlib.pyplot as plt
import pandas as pd
import unitvelo as utv
from os.path import exists
import torch
method = 'latentvelo'

2025-01-07 14:41:23.347161: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-07 14:41:23.348677: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-07 14:41:23.354363: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-07 14:41:23.372082: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-07 14:41:23.400143: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

(Running UniTVelo 0.2.5.2)
2025-01-07 06:41:27


In [ ]:
#dataset name
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina'] #'Hindbrain_GABA_Glio','organogenesis_chondrocyte'
#data path
data_dir = '/data/'
#result path
save_dir = '/result/'

In [ ]:

df_CB= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_CB)
df_IC= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_IC)

Empty DataFrame
Columns: [Mean, Time(s)]
Index: []
Empty DataFrame
Columns: [Mean, Time(s)]
Index: []


In [ ]:

for dataset in datasets:
    print(dataset)
    torch.manual_seed(521)
    adata = sc.read_h5ad(data_dir + dataset +'/'+ f'{dataset}.h5ad')
    os.chdir("/file_path/latentvelo/")
    start = time.time()
    #setup data and train model
    scv.pp.filter_and_normalize(adata, min_shared_counts=20)
    scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
    adata = ltv.utils.anvi_clean_recipe(adata, celltype_key='clusters')
    model = ltv.models.AnnotVAE(observed=2000, latent_dim=30, encoder_hidden=35, zr_dim=2, h_dim=3, 
                                celltypes=len(adata.obs.clusters.unique()))
    epochs, val_ae, val_traj = ltv.train(model, adata, batch_size = 100, learning_rate=1e-2,
                                        epochs=24, name=f'{dataset}_parameters',grad_clip=100) 
    end = time.time()
    latent_adata, adata = ltv.output_results(model, adata, gene_velocity=True, decoded=True)
    scv.tl.velocity_graph(latent_adata, vkey='spliced_velocity')
    fig_title=f'{method}-{dataset}'
    plt_save_dir=save_dir + method +'/'+'CB_IC/'+ f'{dataset}_{method}.pdf'
    scv.pl.velocity_embedding_stream(latent_adata, basis='umap', save = plt_save_dir, vkey='spliced_velocity',color="clusters",title=fig_title,size=200)
     # Calculate performance metrics:
    file = open(data_dir + dataset +'/'+f'{dataset}_groundTruth.pickle' ,'rb')
    ground_truth = pickle.load(file)
    scv.tl.velocity_embedding(latent_adata, vkey='spliced_velocity', basis='pca')
    metrics = utv.evaluate(latent_adata, ground_truth, 'clusters', 'spliced_velocity')
    if exists(save_dir + method +'/'+ '_CBDir_scores.csv'):
        tab = pd.read_csv(save_dir + method +'/'+'_CBDir_scores.csv', index_col = 0)
    else:
        tab_cb = pd.DataFrame(columns = list(metrics['Cross-Boundary Direction Correctness (A->B)'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
        tab_IC = pd.DataFrame(columns = list(metrics['In-cluster Coherence'].keys())  + ['Mean', 'Time(s)'],
                 index = [dataset])
    ##CBDC_scores
    cb_score = [np.mean(metrics['Cross-Boundary Direction Correctness (A->B)'][x])
                for x in metrics['Cross-Boundary Direction Correctness (A->B)'].keys()]
    tab_cb.loc[dataset,:] = cb_score + [np.mean(cb_score), end-start]
    df_CB=df_CB.append(pd.DataFrame([[np.mean(cb_score), end-start]],columns=df_CB.columns,index=[dataset]))
    #tab_cb.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_CBDir_scores.csv')
    ##ICCoh_scores
    IC_score = [np.mean(metrics['In-cluster Coherence'][x])
                for x in metrics['In-cluster Coherence'].keys()]
    tab_IC.loc[dataset,:] = IC_score + [np.mean(IC_score), end-start]

    df_IC=df_IC.append(pd.DataFrame([[np.mean(IC_score), end-start]],columns=df_IC.columns,index=[dataset]))
    #tab_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_ICCoh_scores.csv')
    latent_adata.write_h5ad(save_dir + method +'/' +'CB_IC/'+ f'{dataset}_AnnData_Forscore.h5ad')

In [ ]:
print(df_CB)
print(df_IC)
df_CB.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_CBDir_scores1.csv')
df_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_ICCoh_scores1.csv')